# 6-2절 연습 문제 풀이

이 노트북은 6-2절 연습 문제(6-4 ~ 6-8)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch06/06-02_example.ipynb`를 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

EOS_TOKEN = '<eos>'
WINDOW_SIZE = 5

poem = '엄마야 누나야 강변 살자 ' \
       '뜰에는 반짝이는 금모래빛 ' \
       '뒷문 밖에는 갈잎의 노래 ' \
       '엄마야 누나야 강변 살자'

학습 장치: cuda


## 연습 문제 6-4

> [연습 문제 6-1]에서 만든 어휘 사전을 사용해 역방향 어휘 사전, 데이터셋, 데이터로더를 만들어 보자.
> 각 글자를 토큰으로 하는 경우와 각 어절을 토큰으로 하는 경우 각각 크기가 5인 슬라이딩 윈도우를 사용한다.

In [2]:
# 본문 [코드 6-6]의 MelodyDataset 을 토큰 목록을 받도록 조금 일반화한다
class SequenceDataset(Dataset):
    """토큰 목록에 <eos>를 붙이고 슬라이딩 윈도우로 (입력, 정답) 쌍을 만드는 데이터셋."""

    def __init__(self, tokens, vocab, window_size=WINDOW_SIZE):
        self.vocab = vocab
        self.window_size = window_size
        token_list = list(tokens) + [EOS_TOKEN]
        self.sequence_idx = torch.tensor([vocab[token] for token in token_list])

    def __len__(self):
        return len(self.sequence_idx) - self.window_size + 1

    def __getitem__(self, idx):
        subsequence = self.sequence_idx[idx: idx + self.window_size]
        x = F.one_hot(subsequence[:-1], num_classes=len(self.vocab)).float()
        y = subsequence[-1]      # 정답은 원-핫이 아니라 고유 번호 그대로
        return x, y

def build_vocab(tokens):
    vocab = {EOS_TOKEN: 0}
    for i, token in enumerate(sorted(set(tokens)), start=1):
        vocab[token] = i
    return vocab

char_tokens = list(poem)
word_tokens = poem.split()
char_vocab = build_vocab(char_tokens)
word_vocab = build_vocab(word_tokens)

# 역방향 어휘 사전: 고유 번호 -> 토큰
char_reversed_vocab = {i: token for token, i in char_vocab.items()}
word_reversed_vocab = {i: token for token, i in word_vocab.items()}

char_dataset = SequenceDataset(char_tokens, char_vocab)
word_dataset = SequenceDataset(word_tokens, word_vocab)
char_loader = DataLoader(char_dataset, batch_size=8, shuffle=True)
word_loader = DataLoader(word_dataset, batch_size=4, shuffle=True)

for name, vocab, reversed_vocab, dataset, loader in [
        ('글자', char_vocab, char_reversed_vocab, char_dataset, char_loader),
        ('어절', word_vocab, word_reversed_vocab, word_dataset, word_loader)]:
    x, y = dataset[0]
    batch_x, batch_y = next(iter(loader))
    print(f'[{name} 토큰]')
    print(f'  어휘 사전 크기: {len(vocab)}, 샘플 수: {len(dataset)}')
    print(f'  역방향 어휘 사전 앞 5개: {dict(list(reversed_vocab.items())[:5])}')
    print(f'  샘플 하나: 입력 {tuple(x.shape)} (S, input_size), 정답 {y.item()} -> {reversed_vocab[y.item()]!r}')
    print(f'  배치 하나: 입력 {tuple(batch_x.shape)} (B, S, input_size), 정답 {tuple(batch_y.shape)} (B,)')
    print()

[글자 토큰]
  어휘 사전 크기: 28, 샘플 수: 52
  역방향 어휘 사전 앞 5개: {0: '<eos>', 1: ' ', 2: '갈', 3: '강', 4: '금'}
  샘플 하나: 입력 (4, 28) (S, input_size), 정답 7 -> '누'
  배치 하나: 입력 (8, 4, 28) (B, S, input_size), 정답 (8,) (B,)

[어절 토큰]
  어휘 사전 크기: 12, 샘플 수: 12
  역방향 어휘 사전 앞 5개: {0: '<eos>', 1: '갈잎의', 2: '강변', 3: '금모래빛', 4: '노래'}
  샘플 하나: 입력 (4, 12) (S, input_size), 정답 7 -> '뜰에는'
  배치 하나: 입력 (4, 4, 12) (B, S, input_size), 정답 (4,) (B,)



### 풀이 해설

본문 [코드 6-6]을 거의 그대로 옮기면 된다. 확인할 점은 세 가지다.

**1. 역방향 어휘 사전은 딕셔너리 컴프리헨션 한 줄이면 된다.**
`{i: token for token, i in vocab.items()}`로 키와 값을 맞바꾼다.
이것이 필요한 이유는 모델이 내놓는 것이 **토큰이 아니라 고유 번호**이기 때문이다.
`argmax()`로 얻은 번호를 사람이 읽을 토큰으로 되돌리려면 반대 방향의 사전이 있어야 한다.
어휘 사전의 값이 서로 겹치지 않는 고유 번호이므로 이렇게 뒤집어도 정보가 사라지지 않는다.

**2. 데이터셋이 반환하는 입력은 원-핫, 정답은 번호다.**
출력에서 입력이 `(4, 28)` 형태의 원-핫 텐서인 반면 정답은 스칼라 하나인 것을 확인할 수 있다.
`nn.CrossEntropyLoss`가 정답을 클래스 번호로 받기 때문이다(연습 문제 6-3의 해설 참고).

**3. 데이터로더를 거치면 배치 차원이 앞에 붙는다.**
`(S, input_size)`가 `(B, S, input_size)`가 된다. 본문 p14가 말한 대로,
이 형태가 `nn.RNN`의 `batch_first=True` 설정과 정확히 맞물린다.

한편 **어절 데이터셋은 샘플이 12개뿐**이라 배치 크기를 4 정도로 줄여야 배치가 세 개라도 나온다.
데이터가 이렇게 적으면 학습이 제대로 될 리 없는데, 이것이 6-5에서 확인할 내용의 배경이 된다.

### 문제 검토

- **적절성: 적합.** 6-1절 연습 문제 세 개로 데이터를 준비했으니, 6-2절 첫 문제로 그것을 파이토치 객체로 옮기는 것은 자연스러운 순서다.
  본문 [코드 6-6]을 거의 그대로 재사용할 수 있어 난도도 알맞다.
- **[검토] 역방향 어휘 사전을 여기서 만들게 한 것이 좋다.** 본문에서는 [코드 6-7]에서야 슬쩍 등장하는데,
  이 문제가 데이터 준비 단계로 끌어올려 '예측 결과를 되돌릴 준비'까지 한 묶음으로 보게 한다.
- **[검토] 어절 데이터셋의 크기 문제를 짚어 주면 좋겠다.** 어절 토큰으로는 샘플이 12개뿐이라
  배치 크기를 본문처럼 잡으면 배치가 하나밖에 안 나온다. 실행하면서 당황할 수 있으니
  '데이터 크기에 맞게 배치 크기를 정하라'는 한마디가 있으면 친절하다.

**윤문안**

> **6-4** [연습 문제 6-1]에서 만든 어휘 사전을 사용해 역방향 어휘 사전, 데이터셋, 데이터로더를 만들어 보자.
> 각 글자를 토큰으로 하는 경우와 각 어절을 토큰으로 하는 경우 각각 크기가 5인 슬라이딩 윈도우를 사용하며,
> 배치 크기는 각 데이터셋의 샘플 수를 고려해 정한다.

## 연습 문제 6-5

> [연습 문제 6-4]에서 각 글자를 토큰으로 하는 데이터셋, 데이터로더, 어휘 사전과 역방향 어휘 사전을 사용해
> `MelodyRNN` 모델을 학습한 후, `'엄마야 '`, `'누나야 '`, `'강변살자'` 각각을 마중물 텍스트로 사용해 텍스트를 생성해 보고 다음 질문에 답해 보자.
> - 모델의 성능은 어떠한가? 그렇게 평가한 이유와 함께 정리해 보자.
> - 세 마중물 텍스트 중 예외가 발생한 경우가 있다면 그 이유는 무엇이며, 예외 처리 방법은 무엇일까?

In [3]:
# 본문 [코드 6-5]의 MelodyRNN 모델 클래스
class MelodyRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):                   # (B, S, input_size)
        outputs, _ = self.rnn(x)            # -> (B, S, hidden_size)
        last_hidden = outputs[:, -1, :]     # -> (B, hidden_size)
        return self.fc(last_hidden)         # -> (B, output_size)

def train(model, loader, epochs=200, learning_rate=0.001, verbose_every=50):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, sample_size = 0.0, 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * inputs.size(0)
            sample_size += inputs.size(0)
        history.append(loss_sum / sample_size)
        if epoch % verbose_every == 0 or epoch == 1:
            print(f'  에포크 {epoch:4d} | 훈련 손실 {history[-1]:.4f}')
    return history

torch.manual_seed(SEED)
char_model = MelodyRNN(len(char_vocab), 32, len(char_vocab))
print('글자 토큰 모델 학습')
char_history = train(char_model, char_loader)

글자 토큰 모델 학습


  에포크    1 | 훈련 손실 3.3271


  에포크   50 | 훈련 손실 1.2684


  에포크  100 | 훈련 손실 0.4143


  에포크  150 | 훈련 손실 0.1712


  에포크  200 | 훈련 손실 0.0958


In [4]:
# 본문 [코드 6-7], [코드 6-8]의 예측·생성 함수
def predict_next(model, sequence, vocab, reversed_vocab, device=None):
    if device is None:
        device = torch.device('cpu')
    if len(sequence) < 4:
        raise ValueError('예측에는 최소 4개의 토큰이 필요합니다.')
    sequence = sequence[-4:]
    model.eval()
    sequence_idx = torch.tensor([vocab[token] for token in sequence])
    input_tensor = F.one_hot(sequence_idx, len(vocab)).float().unsqueeze(0).to(device)
    with torch.no_grad():
        predicted_idx = torch.argmax(model(input_tensor)).item()
    return reversed_vocab[predicted_idx]

def generate_sequence(model, start_sequence, generated_length, vocab, reversed_vocab,
                      eos_token=EOS_TOKEN, device=None):
    result = list(start_sequence)
    for _ in range(generated_length):
        next_token = predict_next(model, result[-4:], vocab, reversed_vocab, device)
        result.append(next_token)
        if eos_token is not None and next_token == eos_token:
            break
    return result

PRIMERS = ['엄마야 ', '누나야 ', '강변살자']
for primer in PRIMERS:
    print(f'마중물 {primer!r} (길이 {len(primer)})')
    try:
        generated = generate_sequence(char_model, primer, 40, char_vocab,
                                      char_reversed_vocab, device=device)
        print(f"  생성 결과: {''.join(generated)!r}")
    except Exception as error:
        print(f'  예외 발생: {type(error).__name__}: {error}')
    print()

마중물 '엄마야 ' (길이 4)
  생성 결과: '엄마야 누나야 강변 살자<eos>'

마중물 '누나야 ' (길이 4)
  생성 결과: '누나야 강변 살자<eos>'

마중물 '강변살자' (길이 4)
  생성 결과: '강변살자자 에는는금짝래래 금문래빛 는문 엄래는 누문의 강는 살자이뜰에는 반짝이는'



In [5]:
# 마중물의 각 토큰이 어휘 사전에 있는지 확인한다
for primer in PRIMERS:
    missing = [token for token in primer if token not in char_vocab]
    print(f'{primer!r}: 어휘 사전에 없는 토큰 {missing if missing else "없음"}')

print()
# 원문에 이 네 글자 배열이 실제로 나오는지도 확인한다
for primer in PRIMERS:
    print(f'{primer!r}: 원문에 등장 {"O" if primer in poem else "X"}')

'엄마야 ': 어휘 사전에 없는 토큰 없음
'누나야 ': 어휘 사전에 없는 토큰 없음
'강변살자': 어휘 사전에 없는 토큰 없음

'엄마야 ': 원문에 등장 O
'누나야 ': 원문에 등장 O
'강변살자': 원문에 등장 X


In [6]:
# 예외에 견디도록 예측 함수를 보강한 예시
UNK_TOKEN = '<unk>'

def predict_next_safe(model, sequence, vocab, reversed_vocab, device=None):
    """어휘 사전에 없는 토큰과 너무 짧은 마중물을 모두 처리하는 예측 함수."""
    if device is None:
        device = torch.device('cpu')
    sequence = list(sequence)
    # (1) 마중물이 짧으면 앞을 <eos>로 채워 길이를 맞춘다(패딩 역할)
    if len(sequence) < 4:
        sequence = [EOS_TOKEN] * (4 - len(sequence)) + sequence
    sequence = sequence[-4:]
    # (2) 어휘 사전에 없는 토큰은 <unk> 번호로 대체한다
    unk_idx = vocab.get(UNK_TOKEN, 0)
    sequence_idx = torch.tensor([vocab.get(token, unk_idx) for token in sequence])
    model.eval()
    input_tensor = F.one_hot(sequence_idx, len(vocab)).float().unsqueeze(0).to(device)
    with torch.no_grad():
        predicted_idx = torch.argmax(model(input_tensor)).item()
    return reversed_vocab[predicted_idx]

for primer in PRIMERS + ['강변', '엄마야 누나야 강변 살자 뜰에는']:
    token = predict_next_safe(char_model, primer, char_vocab, char_reversed_vocab, device)
    print(f'{primer!r} -> 다음 글자 {token!r}')

'엄마야 ' -> 다음 글자 '누'
'누나야 ' -> 다음 글자 '강'
'강변살자' -> 다음 글자 '자'
'강변' -> 다음 글자 ' '
'엄마야 누나야 강변 살자 뜰에는' -> 다음 글자 ' '


### 풀이 해설

**첫 번째 물음: 모델의 성능**

생성 결과부터 보자.

```
'엄마야 ' -> '엄마야 누나야 강변 살자<eos>'
'누나야 ' -> '누나야 강변 살자<eos>'
'강변살자' -> '강변살자자 에는는금짝래래 금문래빛 는문 엄래는 누문의 강는 살자이뜰에는 반짝이는'
```

앞의 두 마중물에서는 **원문의 한 행을 정확히 재현하고 `<eos>`까지 제때 내놓는다.**
세 번째에서는 완전히 무너진다. 이 대비가 이 문제의 전부다.

그렇다면 성능이 좋은 것인가? **아니다. 잘된 쪽이 오히려 문제의 증거다.**

`'엄마야 '`로 시작해 `'엄마야 누나야 강변 살자'`가 나온 것은 **시를 통째로 외웠기 때문**이다.
학습 샘플이 52개뿐이고 모델의 파라미터는 그보다 훨씬 많으니 외우는 것이 어려운 일이 아니다.
게다가 `<eos>`까지 정확히 내놓는데, 이는 **`<eos>`가 정답인 샘플이 하나뿐인데도 그 하나를 외웠다**는 뜻이다.

이것이 생성 모델로서 좋은 상태가 아닌 이유는 두 번째 결과가 보여 준다.
`'강변살자'`는 **원문에 없는 네 글자 배열**이다(원문은 `'강변 살자'`로 사이에 공백이 있다).
글자 넷이 모두 어휘 사전에 있는데도 결과가 뒤죽박죽인 것은,
**모델이 규칙을 배운 것이 아니라 본 것만 기억하고 있다**는 증거다.

정리하면 **전형적인 과적합**이다. 55자짜리 시로 한국어를 배울 수는 없으니 당연한 결과이기도 하다.
본문 p17이 <반짝반짝 작은별>에서 `'도레미파'`(학습 데이터에 없는 마중물)로 관찰한 것과 정확히 같은 현상이다.

**평가 방법 자체도 짚어 두자.** 훈련 손실만 보면 잘 내려가서 '학습이 잘됐다'고 읽힌다.
세 마중물의 결과를 직접 읽어 봐야 비로소 한계가 보인다. 본문 p18이 말한 **정성 평가**가 필요한 이유다.

**두 번째 물음: 예외**

위에서 확인한 대로 **세 마중물 모두 예외 없이 동작한다.**
세 마중물의 모든 글자가 어휘 사전에 있고(`'강변살자'`의 네 글자도 모두 있다), 길이도 모두 4라
`predict_next()`의 길이 검사를 통과하기 때문이다.

`'강변살자'`는 **예외가 아니라 품질 저하로 나타나는 경우**다. 글자는 모두 아는데 그 배열을 처음 보는 상황이다.
이 구분이 중요하다. **어휘 사전에 없는 것은 예외를 일으키고, 사전에 있지만 배열을 못 본 것은 엉뚱한 출력을 낳는다.**

**그렇다면 어떤 경우에 예외가 나는가.** 두 가지다.

| 상황 | 발생하는 예외 | 원인 |
|---|---|---|
| 마중물에 어휘 사전에 없는 토큰이 있음(예: `'강변살쟈'`) | `KeyError` | `vocab[token]` 조회 실패 |
| 마중물이 4글자보다 짧음(예: `'강변'`) | `ValueError` | `predict_next()`의 길이 검사 |

**예외 처리 방법**은 위 `predict_next_safe()`에 담았다. 실행 결과를 보면 `'강변'`도 정상 동작한다.

- **모르는 토큰**: 본문 표 6-1의 `<unk>` 토큰을 어휘 사전에 미리 등록해 두고 `vocab.get(token, unk_idx)`로 대체한다.
  단, `<unk>`를 학습 중에 한 번도 보지 못했다면 모델이 그 토큰을 어떻게 다뤄야 할지 모른다는 한계가 남는다.
  실무에서는 **드물게 등장하는 토큰 일부를 일부러 `<unk>`로 바꿔 학습**시켜 이 문제를 줄인다.
- **짧은 마중물**: 앞을 `<pad>`(여기서는 `<eos>`로 대신했다)로 채워 길이를 맞춘다.
  본문 p36의 그림 6-15가 보여 주는 방식과 같은 발상이다.

### 문제 검토

- **적절성: 적합. 특히 두 번째 물음의 설계 의도가 좋다.** 본문은 `<unk>` 토큰을 표 6-1에서 이름만 소개하고
  실제로 쓰지는 않는데, 이 문제가 '언제 필요한가'를 실패 경험으로 알게 한다.
- **★ [검토] 그런데 제시한 세 마중물로는 실제로 예외가 발생하지 않는다.**
  `'엄마야 '`, `'누나야 '`, `'강변살자'`의 **모든 글자가 어휘 사전에 들어 있고 길이도 모두 4**이므로,
  본문 [코드 6-7]의 `predict_next()`를 그대로 쓰면 세 경우 다 정상 실행된다(위 실행 결과로 확인).
  지문은 "예외가 발생한 경우가 **있다면**"이라고 조건을 달아 두었으므로 문장 자체가 틀린 것은 아니지만,
  독자는 예외를 찾으려 한참 헤매다가 결국 못 찾고 넘어갈 가능성이 높다.

  `'강변살자'`를 넣은 의도는 아마 **'강변'과 '살자' 사이의 공백을 뺀 것**을 문제 삼으려던 것으로 보인다.
  이 배열은 학습 데이터에 없어 예측이 엉뚱해지지만, **예외가 아니라 품질 저하**로 나타난다.
  실제로 이쪽이 더 배울 것이 많은 관찰이다.

  → **둘 중 하나로 정리하는 것을 권한다.**
  - (A) **예외가 나는 마중물을 실제로 넣는다.** 예를 들어 `'강변'`(4글자 미만)이나 `'강변살쟈'`(사전에 없는 글자)를 추가한다.
  - (B) **물음의 초점을 예외에서 품질로 옮긴다.** `'강변살자'`가 원문에 없는 배열이라는 점을 짚게 한다.

  (A)와 (B)를 모두 담는 것이 가장 낫다. 아래 윤문안은 그 방향이다.
- **[검토] 마중물 세 개의 선택이 좋다.** `'엄마야 '`는 시의 첫머리, `'누나야 '`는 중간, `'강변살자'`는 변형이다.
  세 결과를 견주면 '학습 데이터에 있는 패턴에서만 잘 동작한다'는 결론에 자연스럽게 이른다.
- **[검토] 첫 번째 물음의 '이유와 함께 정리해 보자'가 중요하다.** 정성 평가를 요구하는 물음이므로,
  본문 p18의 정성 평가 설명과 짝을 이룬다. 좋은 배치다.

**윤문안**

> **6-5** [연습 문제 6-4]에서 각 글자를 토큰으로 하는 데이터셋, 데이터로더, 어휘 사전과 역방향 어휘 사전을 사용해
> `MelodyRNN` 모델을 학습한 후, `'엄마야 '`, `'누나야 '`, `'강변살자'`, `'강변'`, `'강변살쟈'` 각각을 마중물 텍스트로
> 사용해 텍스트를 생성해 보고 다음 질문에 답해 보자.
> - 모델의 성능은 어떠한가? 그렇게 평가한 이유와 함께 정리해 보자.
> - `'강변살자'`는 원문에 없는 글자 배열이다. 생성 결과가 앞의 두 마중물과 어떻게 다른가?
> - 다섯 마중물 텍스트 중 예외가 발생한 경우는 무엇이며 그 이유는 무엇일까? 각각의 예외 처리 방법도 생각해 보자.

## 연습 문제 6-6

> `MelodyRNN` 모델 정의에서 단순 RNN 계층의 출력을 `[:, -1, :]`으로 슬라이싱해 마지막 숨겨진 상태를 골라서
> 완전 연결 계층에 전달한다. 이때 `[:, -2, :]` 슬라이싱을 사용하면 모델이 학습하는 것은 무엇일지 유추해 보자.

### 예상

`outputs[:, k, :]`는 **입력의 앞 `k+1`개 토큰만 본 시점의 숨겨진 상태**다(본문 p11).
입력이 4개일 때 `[:, -1, :]`은 네 개를 다 본 상태, `[:, -2, :]`는 **앞 세 개만 본 상태**다.

그런데 정답은 그대로 '다섯 번째 토큰'이다. 그러므로 `[:, -2, :]`로 바꾸면 모델은
**앞 세 토큰으로 다섯 번째 토큰을 맞히는 문제**를 풀게 된다. 네 번째 토큰은 입력으로 들어가지만
**그 정보가 분류기에 전달되지 않으므로 버려지는 셈**이다.

예상되는 결과는 두 가지다.

1. **성능이 떨어진다.** 바로 앞 토큰(가장 강한 단서)을 못 보고 예측해야 하니 당연하다.
2. **네 번째 토큰 자리의 파라미터도 여전히 학습된다.** 네 번째 입력이 `[:, -1, :]`을 만드는 데 쓰이긴 하지만
   그 값이 손실 계산에 쓰이지 않으므로, 이 부분에는 기울기가 흐르지 않는다.

실제로 확인해 보자.

In [7]:
class MelodyRNNSliced(MelodyRNN):
    """마지막에서 몇 번째 숨겨진 상태를 쓸지 고를 수 있는 모델."""

    def __init__(self, input_size, hidden_size, output_size, slice_index=-1):
        super().__init__(input_size, hidden_size, output_size)
        self.slice_index = slice_index

    def forward(self, x):
        outputs, _ = self.rnn(x)
        return self.fc(outputs[:, self.slice_index, :])

@torch.no_grad()
def accuracy(model, dataset):
    model.eval()
    correct = 0
    for i in range(len(dataset)):
        x, y = dataset[i]
        predicted = model(x.unsqueeze(0).to(device)).argmax().item()
        correct += predicted == y.item()
    return correct / len(dataset) * 100

results = {}
for slice_index in (-1, -2, -3):
    torch.manual_seed(SEED)
    model = MelodyRNNSliced(len(char_vocab), 32, len(char_vocab), slice_index)
    print(f'슬라이싱 [:, {slice_index}, :] 모델 학습')
    history = train(model, char_loader, verbose_every=200)
    results[slice_index] = (history[-1], accuracy(model, char_dataset), model)

print()
print(f'{"슬라이싱":>14} {"실제로 보는 토큰 수":>18} {"최종 훈련 손실":>14} {"훈련 데이터 정확도":>18}')
print('-' * 70)
for slice_index, (loss, acc, _) in results.items():
    visible = WINDOW_SIZE - 1 + slice_index + 1
    print(f'{f"[:, {slice_index}, :]":>14} {visible:18d} {loss:14.4f} {acc:17.2f}%')

슬라이싱 [:, -1, :] 모델 학습
  에포크    1 | 훈련 손실 3.3271


  에포크  200 | 훈련 손실 0.0958
슬라이싱 [:, -2, :] 모델 학습
  에포크    1 | 훈련 손실 3.3351


  에포크  200 | 훈련 손실 0.1160
슬라이싱 [:, -3, :] 모델 학습
  에포크    1 | 훈련 손실 3.3538


  에포크  200 | 훈련 손실 0.2359

          슬라이싱        실제로 보는 토큰 수       최종 훈련 손실         훈련 데이터 정확도
----------------------------------------------------------------------
    [:, -1, :]                  4         0.0958             98.08%
    [:, -2, :]                  3         0.1160             96.15%
    [:, -3, :]                  2         0.2359             88.46%


### 풀이 해설

실행 결과가 예상을 그대로 확인해 준다. **뒤에서 몇 번째를 고르느냐가 곧 '몇 개의 토큰을 보고 예측하느냐'다.**

| 슬라이싱 | 분류기가 보는 숨겨진 상태 | 실제로 활용되는 입력 | 최종 훈련 손실 | 훈련 데이터 정확도 |
|---|---|---|---|---|
| `[:, -1, :]` | 네 번째 토큰까지 본 상태 | 4개 | 0.0958 | 98.08% |
| `[:, -2, :]` | 세 번째 토큰까지 본 상태 | 3개 | 0.1160 | 96.15% |
| `[:, -3, :]` | 두 번째 토큰까지 본 상태 | 2개 | 0.2359 | 88.46% |

뒤로 갈수록 훈련 손실이 커지고 정확도가 떨어진다. **정답 바로 앞의 토큰이 가장 강한 단서**이기 때문이다.

눈여겨볼 것은 **떨어지는 폭이 일정하지 않다**는 점이다. 4개 → 3개에서는 정확도가 2%p 남짓 떨어지는데,
3개 → 2개에서는 8%p 가까이 떨어진다. 글자 두 개만으로는 다음 글자를 짐작할 단서가 턱없이 부족한 것이다.
'문맥을 조금 줄이는 것'과 '문맥이 사라지는 것' 사이에 문턱이 있다는 뜻이기도 하다.

여기서 두 가지를 더 짚어 두면 좋다.

**첫째, 이것은 '윈도우 크기를 줄인 것'과 거의 같다.** `[:, -2, :]`를 쓰는 모델은 사실상 윈도우 크기를 4로 잡고
앞 세 토큰으로 예측하는 모델과 같은 일을 한다. 다만 **불필요한 연산을 더 한다**는 점이 다르다.
네 번째 토큰도 RNN을 한 번 더 통과하지만 그 결과는 손실 계산에 쓰이지 않으므로 그 단계에는 기울기도 흐르지 않는다.
즉 **계산만 버리는 셈이다.**

**둘째, 그렇다고 이 슬라이싱이 언제나 잘못된 것은 아니다.**
6-3절 본문 p36~37의 추가 설명이 정확히 이 아이디어를 쓴다.
띄어쓰기 예측에서 **예측 위치를 입력 샘플의 가운데로 당기면**, 그 뒤의 문자열까지 예측에 반영된다.
이때는 `[:, -1, :]`(윈도우 전체를 본 마지막 상태)을 쓰되 **정답의 위치를 옮기는데**, 발상은 같다.
'무엇을 보고 무엇을 맞힐 것인가'를 설계하는 일이라는 점이 핵심이다.

이 문제의 진짜 소득은 **`outputs`의 두 번째 차원이 시간 축이라는 사실을 체감하는 것**이다.
`(B, S, hidden_size)`에서 `S`가 왜 '시점'인지, 그 축을 자르면 무슨 일이 일어나는지가 몸에 남는다.

### 문제 검토

- **적절성: 적합. 6장에서 가장 잘 만든 문제 중 하나다.** 코드 한 글자(`-1` → `-2`)만 바꾸는 최소한의 변경으로
  `outputs`의 시간 축이라는 개념을 정확히 겨눈다. 본문 p11의 셋째 규칙("`outputs[:, k - 1, :]`은 `k`번째 요소가
  입력된 후의 숨겨진 상태")을 제대로 읽었는지 가려내는 물음이기도 하다.
- **[검토] '유추해 보자'로 끝나 실행을 요구하지 않는다.** 답을 말로만 정리하면 '성능이 떨어진다' 수준에서 멈춘다.
  실제로 돌려 보면 `-1`, `-2`, `-3`이 계단처럼 나빠지는 것이 보여 '보는 토큰 수'라는 해석이 확증된다.
  한 줄만 바꾸면 되므로 실행 부담도 없다.
- **[검토] 6-3절의 추가 설명과 이어 주면 더 좋겠다.** p36~37의 '예측 위치를 앞으로 당기기'가 같은 아이디어의
  긍정적 활용인데, 이 연결을 짚어 주면 문제가 단순한 함정 확인에서 설계 감각으로 확장된다.
  다만 6-2절 시점에서는 아직 안 나온 내용이라, 6-3절 쪽에 참조를 남기는 편이 나을 수도 있다.

**윤문안**

> **6-6** `MelodyRNN` 모델 정의에서 단순 RNN 계층의 출력을 `[:, -1, :]`으로 슬라이싱해 마지막 숨겨진 상태를 골라서
> 완전 연결 계층에 전달한다. 이때 `[:, -2, :]` 슬라이싱을 사용하면 모델이 학습하는 것은 무엇일지 유추해 본 후,
> 실제로 바꿔 학습해 훈련 손실과 생성 결과를 비교해 보자.

## 연습 문제 6-7

> 단순 RNN 계층의 과거 기억 용량은 `hidden_size`로 결정되며, 이 값을 바꾸면 기억 크기와 파라미터 수가 함께 달라진다.
> 그러면 과거를 더 많이 기억한다고 생성 결과가 항상 좋아질까? 이 질문에 대한 답을 `hidden_size`를 4, 8, 16, 32, 64로
> 바꿔 가며 `MelodyRNN` 모델을 학습한 뒤, 학습 로그와 생성 결과를 비교해 찾아 보자.

In [8]:
# 본문 예제와 같은 <반짝반짝 작은별> 멜로디로 실험한다
melody = '도도솔솔라라솔_파파미미레레도_솔솔파파미미레_' \
         '솔솔파파미미레_도도솔솔라라솔_파파미미레레도_'
melody_tokens = list(melody)
melody_vocab = build_vocab(melody_tokens)
melody_reversed_vocab = {i: token for token, i in melody_vocab.items()}
melody_dataset = SequenceDataset(melody_tokens, melody_vocab)
melody_loader = DataLoader(melody_dataset, batch_size=8, shuffle=True)
print(f'어휘 사전: {melody_vocab}')
print(f'샘플 수: {len(melody_dataset)}개')
print(f'원곡: {melody}')

어휘 사전: {'<eos>': 0, '_': 1, '도': 2, '라': 3, '레': 4, '미': 5, '솔': 6, '파': 7}
샘플 수: 45개
원곡: 도도솔솔라라솔_파파미미레레도_솔솔파파미미레_솔솔파파미미레_도도솔솔라라솔_파파미미레레도_


In [9]:
HIDDEN_SIZES = (4, 8, 16, 32, 64)
sweep = {}
for hidden_size in HIDDEN_SIZES:
    torch.manual_seed(SEED)
    model = MelodyRNN(len(melody_vocab), hidden_size, len(melody_vocab))
    history = train(model, melody_loader, epochs=200, verbose_every=200)
    generated = generate_sequence(model, '도도솔솔', 40, melody_vocab,
                                  melody_reversed_vocab, device=device)
    params = sum(p.numel() for p in model.parameters())
    sweep[hidden_size] = (history[-1], accuracy(model, melody_dataset), params, ''.join(generated))

print()
print(f'{"hidden_size":>12} {"파라미터 수":>12} {"최종 훈련 손실":>14} {"훈련 데이터 정확도":>18}')
print('-' * 64)
for hidden_size, (loss, acc, params, _) in sweep.items():
    print(f'{hidden_size:12d} {params:12,d} {loss:14.4f} {acc:17.2f}%')

  에포크    1 | 훈련 손실 2.1488


  에포크  200 | 훈련 손실 0.9915
  에포크    1 | 훈련 손실 2.1134


  에포크  200 | 훈련 손실 0.4614
  에포크    1 | 훈련 손실 2.1374


  에포크  200 | 훈련 손실 0.2752
  에포크    1 | 훈련 손실 2.0624


  에포크  200 | 훈련 손실 0.1915
  에포크    1 | 훈련 손실 2.0660


  에포크  200 | 훈련 손실 0.1525

 hidden_size       파라미터 수       최종 훈련 손실         훈련 데이터 정확도
----------------------------------------------------------------
           4           96         0.9915             68.89%
           8          216         0.4614             84.44%
          16          552         0.2752             91.11%
          32        1,608         0.1915             91.11%
          64        5,256         0.1525             91.11%


In [10]:
print(f'원곡  : {melody}')
print()
for hidden_size, (_, _, _, generated) in sweep.items():
    print(f'h={hidden_size:<3d}: {generated}')

원곡  : 도도솔솔라라솔_파파미미레레도_솔솔파파미미레_솔솔파파미미레_도도솔솔라라솔_파파미미레레도_

h=4  : 도도솔솔솔솔파파미미레레도도솔솔솔솔파파미미레레도도솔솔솔솔파파미미레레도도솔솔솔솔파파
h=8  : 도도솔솔라라솔_파파미미레레도도솔솔라라솔_파파미미레레도도솔솔라라솔_파파미미레레도도
h=16 : 도도솔솔라라솔_파파미미레레도_솔솔파파미미레레도_솔솔파파미미레레도_솔솔파파미미레레
h=32 : 도도솔솔라라솔_파파미미레레도_솔솔파파미미레레도_솔솔파파미미레레도_솔솔파파미미레레
h=64 : 도도솔솔라라솔_파파미미레레도_<eos>


### 풀이 해설

**결론부터 말하면 '항상 좋아지지는 않는다'.** 그런데 그 이유가 흔히 짐작하는 것과 상당히 다르다.

| `hidden_size` | 파라미터 수 | 최종 훈련 손실 | 훈련 데이터 정확도 |
|---|---|---|---|
| 4 | 96 | 0.9915 | 68.89% |
| 8 | 216 | 0.4614 | 84.44% |
| 16 | 552 | 0.2752 | **91.11%** |
| 32 | 1,608 | 0.1915 | **91.11%** |
| 64 | 5,256 | 0.1525 | **91.11%** |

**손실은 끝까지 계속 내려가는데 정확도는 16에서 멈춘다.** 이 어긋남이 첫 번째 관찰거리다.
16에서 64로 가며 파라미터가 열 배로 늘고 손실은 거의 절반이 되지만, 맞히는 샘플 수는 하나도 늘지 않는다.
손실이 더 내려간 것은 **이미 맞히던 것을 더 확신하게 되었다**는 뜻일 뿐이다.
남은 오답은 데이터 자체가 모순이라 어떤 모델도 맞힐 수 없는 샘플이다
(같은 네 음 뒤에 서로 다른 음이 오는 경우. 6-2 해설에서 본 것과 같은 상황이다).

생성 결과를 나란히 놓고 보면 더 분명해진다. 마중물은 모두 `'도도솔솔'`이다.

```
원곡  : 도도솔솔라라솔_파파미미레레도_솔솔파파미미레_솔솔파파미미레_도도솔솔라라솔_파파미미레레도_
h=4   : 도도솔솔솔솔파파미미레레도도솔솔솔솔파파미미레레도도솔솔솔솔파파...
h=8   : 도도솔솔라라솔_파파미미레레도도솔솔라라솔_파파미미레레도도솔솔...
h=16  : 도도솔솔라라솔_파파미미레레도_솔솔파파미미레레도_솔솔파파미미레레...
h=32  : 도도솔솔라라솔_파파미미레레도_솔솔파파미미레레도_솔솔파파미미레레...
h=64  : 도도솔솔라라솔_파파미미레레도_<eos>
```

**h=4**: `라라솔`을 아예 못 낸다. 기억 용량이 부족해 원곡의 패턴을 담지 못하는 **과소적합**이다.

**h=8**: 첫 두 마디는 맞지만 `_` 다음에 바로 `도도솔솔`로 돌아가 무한 반복에 빠진다.

**h=16, h=32**: 원곡의 세 번째 마디까지 따라가다 `레레도_` 부근에서 앞으로 되돌아가 역시 반복에 빠진다.
둘의 생성 결과가 **사실상 같다**는 점이 중요하다. 파라미터가 세 배 차이인데 결과가 다르지 않다.

**h=64**: 여기서 가장 흥미로운 일이 일어난다. **두 마디만 내고 `<eos>`로 끝내 버린다.**
원곡은 여섯 마디인데 4분의 1만 생성하고 만 것이다.
왜 그럴까? 원곡에서 `파파미미레레도_`는 **두 번 나오는데**, 두 번째 것 다음이 곡의 끝(`<eos>`)이다.
용량이 큰 모델은 이 `<eos>` 샘플 하나까지 확실하게 외워 버려서, **첫 번째 등장에서도 `<eos>`를 내놓는다.**
즉 h=64는 '더 잘 기억한 탓에 더 일찍 끝내는' 모델이 되었다.

**손실이 가장 낮은 모델이 가장 나쁜 생성 결과를 냈다.** 본문 p18이 "학습 데이터에서 측정한 손실값이
낮게 나오더라도 실제 생성 능력이 좋다고 단정할 수 없다"고 한 것이 이보다 선명할 수 없는 사례다.

**파라미터 수의 증가 방식도 짚어 두자.** 표를 보면 `hidden_size`가 두 배가 될 때 파라미터가
**96 → 216 → 552 → 1,608 → 5,256**으로, 두 배가 아니라 **세 배 안팎씩** 뛴다.
단순 RNN에는 숨겨진 상태를 숨겨진 상태로 옮기는 `hidden_size × hidden_size` 가중치 행렬이 있어
파라미터가 `hidden_size`의 **제곱에 비례하는 항**을 포함하기 때문이다.
"값이 클수록 더 많은 과거 정보를 담을 수 있지만, 파라미터 수도 함께 늘어난다"는 본문 표 6-2의 설명이
**'함께'가 아니라 '제곱으로'**라는 것을 알고 나면 하이퍼파라미터를 고를 때의 감각이 달라진다.

정리하면, `hidden_size`는 **데이터의 복잡도에 맞춰야 하는 값**이지 크면 클수록 좋은 값이 아니다.
이 데이터에서는 16 정도가 적당하고, 본문이 고른 32는 여유를 조금 둔 선택이다.

### 문제 검토

- **적절성: 적합. 물음의 표현이 특히 좋다.** "과거를 더 많이 기억한다고 생성 결과가 항상 좋아질까?"는
  독자가 당연히 '그렇다'고 답할 법한 물음을 정면으로 던져 예상을 흔든다.
- **[검토] '학습 로그와 생성 결과를 비교하라'는 요구가 이 문제의 핵심이다.** 둘을 함께 보게 한 덕분에
  **손실은 계속 내려가는데 생성 품질은 나아지지 않는** 어긋남을 발견할 수 있다.
  손실만 보게 했다면 '클수록 좋다'는 잘못된 결론에 이르렀을 것이다. 잘 설계된 물음이다.
- **[검토] 다섯 값의 범위가 알맞다.** 4는 확실히 부족하고 64는 확실히 과한 지점이라,
  본문이 고른 32가 어디쯤인지 감이 잡힌다. 2의 거듭제곱으로 두 배씩 올린 것도 비교하기에 좋다.
- **[검토] 파라미터 수도 함께 세어 보게 하면 좋겠다.** 지문이 "파라미터 수가 함께 달라진다"고 언급하면서도
  세어 보라고는 하지 않는다. 실제로 세어 보면 **제곱으로 늘어난다**는 것을 알게 되는데,
  이는 표 6-2의 설명만으로는 얻기 어려운 정보다.

**윤문안**

> **6-7** 단순 RNN 계층의 과거 기억 용량은 `hidden_size`로 결정되며, 이 값을 바꾸면 기억 크기와 파라미터 수가 함께 달라진다.
> 그러면 과거를 더 많이 기억한다고 생성 결과가 항상 좋아질까? 이 질문에 대한 답을 `hidden_size`를 4, 8, 16, 32, 64로
> 바꿔 가며 `MelodyRNN` 모델을 학습한 뒤, 학습 로그와 생성 결과를 비교해 찾아 보자.
> 각 모델의 파라미터 수도 함께 세어 `hidden_size`가 두 배가 될 때 파라미터가 몇 배가 되는지 확인해 보자.

## 연습 문제 6-8 [도전 문제]

> `MelodyRNN`을 <반짝반짝 작은별>의 멜로디만으로 학습하는 대신 다음 세 동요의 멜로디를 추가해
> 네 동요의 음으로 학습 데이터를 구성해 모델을 만들어 본 후, 다음 질문에 답해 보자.
> - `<eos>`가 정답인 샘플의 개수는 몇 개로 늘어날까?
> - 한 곡만 학습한 모델과 비교했을 때 생성된 멜로디의 자연스러움에는 어떤 변화가 있는가?
> - `<eos>` 토큰을 예측해 정해진 길이보다 짧게 생성하는 경우가 나타났는가? 만약 그렇지 않다면 학습 데이터를 어떤 방향으로 보강해야 할까?

In [11]:
SONGS = {
    '반짝반짝 작은별': melody,
    '산토끼': '솔_미미솔미도_레_미레도미솔_도솔도솔도솔미_솔_레파미레도_',
    '학교종': '솔솔라라솔솔미_솔솔미미레_솔솔라라솔솔미_솔미레미도_',
    '노는 게 제일 좋아': '솔미도레솔솔_솔미도레라솔_라라시_솔솔라_미라미도미레레_',
}

class MultiSongDataset(Dataset):
    """여러 곡을 각각 <eos>로 끝내고, 곡 경계를 넘지 않도록 슬라이딩 윈도우를 적용한다."""

    def __init__(self, songs, vocab, window_size=WINDOW_SIZE):
        self.vocab = vocab
        self.samples = []
        for song in songs:
            token_list = list(song) + [EOS_TOKEN]
            idx = [vocab[token] for token in token_list]
            for i in range(len(idx) - window_size + 1):
                self.samples.append(torch.tensor(idx[i: i + window_size]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        subsequence = self.samples[idx]
        x = F.one_hot(subsequence[:-1], num_classes=len(self.vocab)).float()
        return x, subsequence[-1]

all_tokens = ''.join(SONGS.values())
multi_vocab = build_vocab(all_tokens)
multi_reversed_vocab = {i: token for token, i in multi_vocab.items()}
multi_dataset = MultiSongDataset(list(SONGS.values()), multi_vocab)
multi_loader = DataLoader(multi_dataset, batch_size=8, shuffle=True)

print(f'어휘 사전 크기: {len(multi_vocab)} (한 곡일 때 {len(melody_vocab)})')
print(f'어휘 사전: {multi_vocab}')
print(f'전체 샘플 수: {len(multi_dataset)}개 (한 곡일 때 {len(melody_dataset)}개)')

eos_idx = multi_vocab[EOS_TOKEN]
eos_count = sum(1 for s in multi_dataset.samples if s[-1].item() == eos_idx)
print(f'정답이 <eos>인 샘플: {eos_count}개 (한 곡일 때 1개)')

어휘 사전 크기: 9 (한 곡일 때 8)
어휘 사전: {'<eos>': 0, '_': 1, '도': 2, '라': 3, '레': 4, '미': 5, '솔': 6, '시': 7, '파': 8}
전체 샘플 수: 126개 (한 곡일 때 45개)
정답이 <eos>인 샘플: 4개 (한 곡일 때 1개)


In [12]:
torch.manual_seed(SEED)
multi_model = MelodyRNN(len(multi_vocab), 32, len(multi_vocab))
print('네 곡 학습')
multi_history = train(multi_model, multi_loader, epochs=300, verbose_every=100)
print(f'훈련 데이터 정확도: {accuracy(multi_model, multi_dataset):.2f}%')

네 곡 학습
  에포크    1 | 훈련 손실 2.1778


  에포크  100 | 훈련 손실 0.6418


  에포크  200 | 훈련 손실 0.2905


  에포크  300 | 훈련 손실 0.2072
훈련 데이터 정확도: 90.48%


In [13]:
PRIMERS_MELODY = ['도도솔솔', '솔_미미', '솔솔라라', '솔미도레', '도레미파']
print('마중물별 생성 결과 (최대 40음, <eos>가 나오면 조기 종료)')
for primer in PRIMERS_MELODY:
    generated = generate_sequence(multi_model, primer, 40, multi_vocab,
                                  multi_reversed_vocab, device=device)
    stopped = generated[-1] == EOS_TOKEN
    print(f"  {primer!r:>10} -> {''.join(generated)}")
    print(f'{"":>14}(생성 길이 {len(generated)}, <eos>로 종료: {"예" if stopped else "아니오"})')

마중물별 생성 결과 (최대 40음, <eos>가 나오면 조기 종료)
      '도도솔솔' -> 도도솔솔라라솔_파파미미레레도_<eos>
              (생성 길이 17, <eos>로 종료: 예)
      '솔_미미' -> 솔_미미솔미도_레_미레도미솔_도솔도솔도솔도솔도솔도솔도솔도솔도솔도솔도솔도솔도솔도솔
              (생성 길이 44, <eos>로 종료: 아니오)
      '솔솔라라' -> 솔솔라라솔_파파미미레레도_<eos>
              (생성 길이 15, <eos>로 종료: 예)
      '솔미도레' -> 솔미도레라솔_라라시_솔솔라_미라미도미레레_<eos>
              (생성 길이 24, <eos>로 종료: 예)
      '도레미파' -> 도레미파_도미레레_<eos>
              (생성 길이 11, <eos>로 종료: 예)


### 풀이 해설

**첫 번째 물음: `<eos>`가 정답인 샘플의 개수**

**곡의 수만큼, 즉 4개가 된다.** 실행 결과에서 확인된다(한 곡일 때 1개 → 네 곡일 때 4개).

다만 여기에는 **구현상의 갈림길**이 하나 있다. 네 곡을 어떻게 이어 붙이느냐에 따라 답이 달라진다.

- **곡을 하나의 긴 문자열로 이어 붙이고 끝에만 `<eos>`를 붙이면** → `<eos>` 샘플은 여전히 **1개**다.
  게다가 곡 경계를 넘나드는 엉터리 샘플(앞 곡의 끝 + 뒤 곡의 시작)이 만들어진다.
- **곡마다 따로 `<eos>`를 붙이고 곡 안에서만 윈도우를 굴리면** → `<eos>` 샘플이 **4개**가 되고 경계 문제도 없다.

위 `MultiSongDataset`은 두 번째 방식이다. 전체 샘플 수는 45개에서 126개로 늘었다.

어휘 사전도 8에서 **9**로 하나 늘었다. <노는 게 제일 좋아>의 `라라시`에 **'시'**가 새로 등장하기 때문이다.

**두 번째 물음: 자연스러움의 변화**

마중물을 바꿔 가며 생성한 결과가 흥미롭다.

```
'도도솔솔'(반짝반짝 작은별) -> 도도솔솔라라솔_파파미미레레도_<eos>
'솔_미미'(산토끼)          -> 솔_미미솔미도_레_미레도미솔_도솔도솔도솔도솔도솔...
'솔솔라라'(학교종)          -> 솔솔라라솔_파파미미레레도_<eos>
'솔미도레'(노는 게 제일 좋아) -> 솔미도레라솔_라라시_솔솔라_미라미도미레레_<eos>
'도레미파'(어느 곡에도 없음)  -> 도레미파_도미레레_<eos>
```

두 방향의 변화가 동시에 일어난다.

*좋아지는 쪽*: **곡의 정체성을 어느 정도 유지한다.** `'솔미도레'`로 시작하면 <노는 게 제일 좋아>의 흐름을
거의 그대로 이어 가고, `'솔_미미'`도 <산토끼>의 앞부분을 정확히 재현한다.
학습 샘플이 45개에서 126개로 늘어난 덕분이다.

*나빠지는 쪽*: **곡 사이를 갈아타는 현상이 나타난다.** `'솔솔라라'`(학교종)로 시작했는데
`솔_파파미미레레도_`로 이어져 <반짝반짝 작은별>로 넘어가 버린다.
윈도우가 4음뿐이라 **모델이 '지금 어느 곡을 연주 중인지' 기억할 방법이 없기 때문**이다.
네 음이 겹치는 순간 다른 곡으로 옮겨 탄다.

즉 **'자연스러움'이 두 가지 뜻으로 갈린다.** 음의 이어짐은 자연스럽지만 곡의 일관성은 보장되지 않는다.
어느 쪽을 중시하느냐에 따라 평가가 달라지는데, 이것이 본문 p18이 말한 생성 모델 평가의 어려움이다.

**세 번째 물음: `<eos>`로 종료되는가**

**그렇다. 다섯 마중물 중 네 개가 `<eos>`로 종료되었다.** 한 곡만 학습했을 때와 뚜렷이 달라진 점이다.
`<eos>` 샘플이 1개에서 4개로 늘어난 것만으로 모델이 '끝'을 판단하기 시작한 것이다.

다만 **끝내는 지점이 옳은지는 별개 문제다.** `'도도솔솔'`로 시작한 결과는 두 마디만에 끝나는데,
원곡은 여섯 마디다. 6-7에서 h=64 모델이 보인 것과 같은 현상으로,
`파파미미레레도_`가 곡 안에서 두 번 나오는데 모델이 그 뒤를 늘 `<eos>`로 이어 버리는 것이다.

반대로 `'솔_미미'`(산토끼)로 시작한 경우는 **끝내지 못하고 `도솔`을 무한 반복한다.**
<산토끼>의 끝 패턴(`레파미레도_`)에 이르지 못하고 도중에 다른 곡의 흐름으로 빠졌기 때문이다.

즉 **'끝낼 줄 알게 되었지만 제때 끝내지는 못한다'**가 정확한 평가다.

**보강 방향**은 세 가지다.

1. **곡 수를 대폭 늘린다.** 100곡이면 `<eos>` 샘플도 100개가 된다. 가장 정공법이다.
2. **곡의 끝 패턴을 다양하게 확보한다.** 동요는 대체로 '도'로 끝나므로, 끝음이 다양한 곡을 섞으면
   모델이 '어떤 음 뒤에 끝이 오는가'가 아니라 '어떤 흐름 뒤에 끝이 오는가'를 배울 수 있다.
3. **윈도우를 키운다.** 위에서 본 두 실패(너무 일찍 끝내기, 곡 갈아타기)는 **모두 윈도우가 4음뿐이라서** 생긴다.
   8~16음으로 늘리면 '곡의 어느 지점인지'를 모델이 알 수 있어 두 문제가 함께 줄어든다.

본문 p17이 "충분한 양의 데이터와 충분한 길이의 입력 샘플로 학습해야 `<eos>` 토큰이 제 역할을 할 수 있다"고
한 것이 1번과 3번에 해당한다. 이 실험은 그 말을 **절반만 따랐을 때 어떤 일이 생기는지**를 보여 준다.

### 문제 검토

- **적절성: 도전 문제로 적합하고, 세 물음의 배치가 훌륭하다.** 계산으로 답할 수 있는 물음(`<eos>` 개수) →
  정성 평가가 필요한 물음(자연스러움) → 실패를 예상하고 해법을 묻는 물음(보강 방향) 순서로 난도가 오른다.
  특히 세 번째 물음의 "**만약 그렇지 않다면**"이 뛰어나다. 실패할 수도 있음을 미리 열어 두어 독자가 자기 구현을 의심하지 않게 한다.
  실제로 돌려 보면 `<eos>` 종료가 **나타난다**(다섯 마중물 중 네 개). 다만 끝내는 지점이 원곡과 맞지 않아,
  '되는가/안 되는가'가 아니라 '어떻게 되는가'를 보게 되는 점이 오히려 이 물음의 수확이다.
- **★ [검토] 첫 번째 물음에 숨은 설계 판단이 있는데 지문이 이를 드러내지 않는다.**
  네 곡을 **한 문자열로 이어 붙이면 `<eos>` 샘플이 1개**이고, **곡마다 따로 `<eos>`를 붙이면 4개**다.
  후자가 맞지만, 전자로 구현하면 곡 경계를 넘나드는 잘못된 샘플까지 만들어진다.
  독자가 이 갈림길을 의식하지 못하고 넘어갈 수 있으므로, 물음에 한 줄 덧붙이면 좋겠다.
- **[검토] 세 곡의 선택이 좋다.** 세 곡 모두 <반짝반짝 작은별>과 음역이 겹치되 '시'가 새로 등장한다
  (<노는 게 제일 좋아>의 `라라시`). 어휘 사전 크기가 한 토큰 늘어나는 것도 관찰거리다.
- **[검토] 두 번째 물음의 '자연스러움'이 모호하다.** 곡 사이를 넘나드는 현상을 어떻게 평가할지에 따라
  답이 정반대가 된다. 이 모호함 자체가 생성 모델 평가의 본질이므로 그대로 두어도 좋지만,
  "어느 곡의 멜로디인지 알아볼 수 있는지도 함께 살펴보자" 정도로 관찰 지점을 하나 주면 답이 풍성해진다.

**윤문안**

> **6-8** [도전 문제] `MelodyRNN`을 <반짝반짝 작은별>의 멜로디만으로 학습하는 대신 다음 세 동요의 멜로디를 추가해
> 네 동요의 음으로 학습 데이터를 구성해 모델을 만들어 본 후, 다음 질문에 답해 보자.
> (세 곡은 그대로)
> - 네 곡을 하나의 순차 데이터로 이어 붙이는 방법과 곡마다 따로 다루는 방법 중 어느 쪽이 알맞을까?
>   그리고 `<eos>`가 정답인 샘플의 개수는 몇 개로 늘어날까?
> - 한 곡만 학습한 모델과 비교했을 때 생성된 멜로디의 자연스러움에는 어떤 변화가 있는가?
>   생성된 멜로디가 어느 곡의 것인지 알아볼 수 있는지도 함께 살펴보자.
> - `<eos>` 토큰을 예측해 정해진 길이보다 짧게 생성하는 경우가 나타났는가? 만약 그렇지 않다면 학습 데이터를 어떤 방향으로 보강해야 할까?